#### the patterns we need 

pd.merge(left, right, on='key')                       # the join pattern

pd.merge(left, right, on='key', how='left')           # keep every row on the left

pd.to_datetime(column, format='%Y-%m-%d')             # the parsing pattern

column.dt.year                                        # and its accessors

df.pivot_table(index=, columns=, values=)             # the pivot pattern

df.groupby('key')['column'].agg(['count', 'mean'])

df.sort_values('column', ascending=False).head(n)

In [1]:
import pandas as pd

base = 'https://eds-217-essential-python.github.io/data/'

temp = pd.read_csv(base + 'monthly_temperature_data.csv')
co2 = pd.read_csv(base + 'monthly_co2_concentration.csv')


In [2]:
temp.shape

(1736, 2)

In [3]:
temp.min()

Date              1880-01-01
MonthlyAnomaly         -0.82
dtype: object

In [4]:
temp.max()

Date              2024-08-01
MonthlyAnomaly          1.48
dtype: object

In [5]:
co2.shape

(796, 2)

In [6]:
co2.min()

Date                1958-04-01
CO2Concentration        312.42
dtype: object

In [7]:
co2.max()

Date                2024-07-01
CO2Concentration        426.91
dtype: object

### Part 1: Making one table out of two

#### 1. How many rows does each table have, and what columns? What is the earliest and latest Date in each? (.min() and .max() on the Date column will do it, because these dates are text in %Y-%m-%d form and text in that form sorts correctly.)


    - temp the min is 1880-01-01 and the max is 2024-08-01

    - co2 the min is 1958-04-01 and the max is 2024-07-01

In [8]:
pd.merge(co2, temp, on = 'Date')

,Date,CO2Concentration,MonthlyAnomaly
0,1958-04-01,317.45,0.01
1,1958-05-01,317.51,0.06
2,1958-06-01,317.27,-0.08
3,1958-07-01,315.87,0.05
4,1958-08-01,314.93,-0.05
...,...,...,...
791,2024-03-01,425.38,1.40
792,2024-04-01,426.57,1.32
793,2024-05-01,426.90,1.16
794,2024-06-01,426.91,1.25


#### 2. Merge the two tables on Date, with the default how=. How many rows come back

    - There are 796 rows.

In [9]:
left = pd.merge(temp, co2, on = 'Date', how = 'left')

left.isnull().sum()




Date                  0
MonthlyAnomaly        0
CO2Concentration    940
dtype: int64

#### 3. Merge them again with how='left', putting temp on the left. How many rows now, and how many nulls, in which column?

    - There are 1736 rows with 940 NULL values in the CO2Concentration.

#### 4. In a markdown cell: the two merges differ by 940 rows. Provide a sentence explaining what those 940 rows contain (your answer to question 1 should already have this)

    - The 940 rows contain the values from the temp but not CO2.

### Part 2: Making the data real

In [10]:
climate = pd.merge(co2, temp, on = 'Date').copy()
climate['date'] = pd.to_datetime(climate['Date'], format = '%Y-%m-%d')

climate.head()

climate['date'].dtype

dtype('<M8[ns]')

#### 6. Copy the inner merge into a table called climate, ending the line with .copy(). Then parse its Date column into a new column called date. Demonstrate that the parse worked by printing the new column’s dtype.

    - here :)

In [11]:
climate['year'] = climate['date'].dt.year
climate['month']= climate['date'].dt.month

climate.head()

,Date,CO2Concentration,MonthlyAnomaly,date,year,month
0,1958-04-01,317.45,0.01,1958-04-01,1958,4
1,1958-05-01,317.51,0.06,1958-05-01,1958,5
2,1958-06-01,317.27,-0.08,1958-06-01,1958,6
3,1958-07-01,315.87,0.05,1958-07-01,1958,7
4,1958-08-01,314.93,-0.05,1958-08-01,1958,8


#### 7. Add year and month columns using the .dt accessors.

    - look up


In [12]:
climate.groupby('year').agg({
    'month': 'count',
    'MonthlyAnomaly': 'mean',
    'CO2Concentration': 'mean'
})



,month,MonthlyAnomaly,CO2Concentration
year,,,
1958,9,0.004444,315.184444
1959,12,0.030833,315.981667
1960,12,-0.025000,316.908333
1961,12,0.057500,317.643333
1962,12,0.030833,318.453333
...,...,...,...
2020,12,1.009167,414.214167
2021,12,0.848333,416.414167
2022,12,0.893333,418.528333


#### 8. Build a table of annual means: group by year and report the count of months, the mean of MonthlyAnomaly, and the mean of CO2Concentration. Show the first three rows and the last three rows.


     - there

In [13]:
full_years = climate.groupby('year')['month'].count() == 12

flat_month =full_years.reset_index()

good_years = flat_month[flat_month['month']]['year'].tolist()

full_years = climate[climate['year'].isin(good_years)]

full_years

,Date,CO2Concentration,MonthlyAnomaly,date,year,month
9,1959-01-01,315.58,0.08,1959-01-01,1959,1
10,1959-02-01,316.49,0.07,1959-02-01,1959,2
11,1959-03-01,316.65,0.18,1959-03-01,1959,3
12,1959-04-01,317.72,0.16,1959-04-01,1959,4
13,1959-05-01,318.29,0.04,1959-05-01,1959,5
...,...,...,...,...,...,...
784,2023-08-01,419.68,1.19,2023-08-01,2023,8
785,2023-09-01,418.50,1.48,2023-09-01,2023,9
786,2023-10-01,418.82,1.34,2023-10-01,2023,10
787,2023-11-01,420.46,1.42,2023-11-01,2023,11


#### 9. Two of the sixty-seven years in the table are not twelve months long. Which two, and why? Filter the data into a new table called full_years. How many years survive?


    - 1958 and 2024. X years survived.

### Part 3: The size of the annual cycle

In [14]:
co2_wide = full_years.pivot_table(index = 'year', columns = 'month', values = 'CO2Concentration')


In [15]:
co2_wide

month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
1959,315.58,316.49,316.65,317.72,318.29,318.15,316.54,314.80,313.84,313.33,314.81,315.58
1960,316.43,316.98,317.58,319.03,320.03,319.58,318.18,315.90,314.17,313.83,315.00,316.19
1961,316.89,317.70,318.54,319.48,320.58,319.77,318.56,316.79,314.99,315.31,316.10,317.01
1962,317.94,318.55,319.68,320.57,321.02,320.62,319.61,317.40,316.24,315.42,316.69,317.70
1963,318.74,319.07,319.86,321.38,322.24,321.49,319.74,317.77,316.21,315.99,317.07,318.35
...,...,...,...,...,...,...,...,...,...,...,...,...
2019,411.03,411.96,412.18,413.54,414.86,414.15,411.96,410.17,408.76,408.74,410.47,411.97
2020,413.59,414.32,414.72,416.42,417.28,416.58,414.58,412.76,411.50,411.49,413.10,414.23
2021,415.49,416.72,417.61,419.01,419.09,418.93,416.90,414.42,413.26,413.90,414.97,416.67


#### 10. Build a wide table with year down the rows, month across the columns, and CO2Concentration in the cells. Show the first three rows and the last three rows

    - LOOK

In [16]:
full_years

,Date,CO2Concentration,MonthlyAnomaly,date,year,month
9,1959-01-01,315.58,0.08,1959-01-01,1959,1
10,1959-02-01,316.49,0.07,1959-02-01,1959,2
11,1959-03-01,316.65,0.18,1959-03-01,1959,3
12,1959-04-01,317.72,0.16,1959-04-01,1959,4
13,1959-05-01,318.29,0.04,1959-05-01,1959,5
...,...,...,...,...,...,...
784,2023-08-01,419.68,1.19,2023-08-01,2023,8
785,2023-09-01,418.50,1.48,2023-09-01,2023,9
786,2023-10-01,418.82,1.34,2023-10-01,2023,10
787,2023-11-01,420.46,1.42,2023-11-01,2023,11


In [17]:
co2_wide

month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
1959,315.58,316.49,316.65,317.72,318.29,318.15,316.54,314.80,313.84,313.33,314.81,315.58
1960,316.43,316.98,317.58,319.03,320.03,319.58,318.18,315.90,314.17,313.83,315.00,316.19
1961,316.89,317.70,318.54,319.48,320.58,319.77,318.56,316.79,314.99,315.31,316.10,317.01
1962,317.94,318.55,319.68,320.57,321.02,320.62,319.61,317.40,316.24,315.42,316.69,317.70
1963,318.74,319.07,319.86,321.38,322.24,321.49,319.74,317.77,316.21,315.99,317.07,318.35
...,...,...,...,...,...,...,...,...,...,...,...,...
2019,411.03,411.96,412.18,413.54,414.86,414.15,411.96,410.17,408.76,408.74,410.47,411.97
2020,413.59,414.32,414.72,416.42,417.28,416.58,414.58,412.76,411.50,411.49,413.10,414.23
2021,415.49,416.72,417.61,419.01,419.09,418.93,416.90,414.42,413.26,413.90,414.97,416.67


#### 11. The wide table holds two different patterns at once. Read it down a single column, then read it across a single row. In a markdown cell, describe both in one sentence each.

    - 

In [18]:
year_1959 = co2_wide.loc[1959].max() - co2_wide.loc[1959].min()
year_2023 = co2_wide.loc[2023].max() - co2_wide.loc[2023].min()

print(year_1959)
print(year_2023)

4.960000000000036
5.5


#### 12. For every year, the difference between its largest monthly value and its smallest is the size of the annual cycle. Compute that difference for two specific years, 1959 and 2023, and report both. (You will need two lines of code; one for each year)

     - Yes